<a href="https://colab.research.google.com/github/bwliauw/Flow_Cytometry_Analysis/blob/main/notebooks/BindCraft_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BindCraft: Protein binder design

<img src="https://github.com/martinpacesa/BindCraft/blob/main/pipeline.png?raw=true">

Simple binder design pipeline using AlphaFold2 backpropagation, MPNN, and PyRosetta. Select your target and let the script do the rest of the work and finish once you have enough designs to order!

The designs will be saved on your Google Drive under BindCraft/[design_name]/ and you can continue running the design pipeline if the session times out and it will continue adding new designs.

In [1]:
#@title Installation
%%time
import os, time, gc, io
import contextlib
import json
from datetime import datetime
from ipywidgets import HTML, VBox
from IPython.display import display

if not os.path.isfile("bindcraft/params/done.txt"):
  print("Installing required BindCraft components")

  print("Pulling BindCraft code from Github")
  os.makedirs('/content/bindcraft/', exist_ok=True)
  !git clone https://github.com/martinpacesa/BindCraft /content/bindcraft/
  os.system("chmod +x /content/bindcraft/functions/dssp")
  os.system("chmod +x /content/bindcraft/functions/DAlphaBall.gcc")

  print("Installing ColabDesign")
  os.system("(mkdir bindcraft/params; apt-get install aria2 -qq; \
  aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; \
  tar -xf alphafold_params_2022-12-06.tar -C bindcraft/params; touch bindcraft/params/done.txt )&")
  os.system("pip install git+https://github.com/sokrypton/ColabDesign.git")
  # for debugging purposes
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")

  print("Installing PyRosetta")
  os.system("pip install pyrosetta_installer")
  with contextlib.redirect_stdout(io.StringIO()):
    import pyrosetta_installer
    pyrosetta_installer.install_pyrosetta(serialization=True)

  # download params
  if not os.path.isfile("bindcraft/params/done.txt"):
    print("downloading AlphaFold params")
    while not os.path.isfile("bindcraft/params/done.txt"):
      time.sleep(5)

  print("BindCraft installation is finished, ready to run!")
else:
  print("BindCraft components already installed, ready to run!")

Installing required BindCraft components
Pulling BindCraft code from Github
Cloning into '/content/bindcraft'...
remote: Enumerating objects: 449, done.
remote: Counting objects: 100% (215/215), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 449 (delta 197), reused 160 (delta 160), pack-reused 234 (from 1)
Receiving objects: 100% (449/449), 5.66 MiB | 19.50 MiB/s, done.
Resolving deltas: 100% (287/287), done.
Installing ColabDesign
Installing PyRosetta
BindCraft installation is finished, ready to run!
CPU times: user 27.9 ms, sys: 8.94 ms, total: 36.9 ms
Wall time: 4min 5s


In [2]:
#@title Mount your Google Drive to save design results
from google.colab import drive
drive.mount('/content/drive')
currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Google drive mounted at: {currenttime}")

bindcraft_google_drive = '/content/drive/My Drive/BindCraft/'
os.makedirs(bindcraft_google_drive, exist_ok=True)
print("BindCraft folder successfully created in your drive!")

Mounted at /content/drive
Google drive mounted at: 2026-05-07 04:14:20
BindCraft folder successfully created in your drive!


In [3]:
#@title Binder design settings
# @markdown ---
# @markdown Enter path where to save your designs. We recommend to save on Google drive so that you can continue generating at any time.
design_path = "/content/drive/My Drive/Code/Protein Design/01.5_bindcraft_output/PD1" # @param {"type":"string","placeholder":"/content/drive/MyDrive/BindCraft/PDL1/"}

# @markdown Enter the name that should be prefixed to your binders (generally target name).
binder_name = "PD1" # @param {"type":"string","placeholder":"PDL1"}

# @markdown The path to the .pdb structure of your target. Can be an experimental or AlphaFold2 structure. We recommend trimming the structure to as small as needed, as the whole selected chains will be backpropagated through the network and can significantly increase running times.
starting_pdb = "/content/drive/My Drive/Code/Protein Design/01_target_preparation/PD1/4ZQK.pdb" # @param {"type":"string","placeholder":"/content/bindcraft/example/PDL1.pdb"}

# @markdown Which chains of your PDB to target? Can be one or multiple, in a comma-separated format. Other chains will be ignored during design.
chains = "A" # @param {"type":"string","placeholder":"A,C"}

# @markdown What positions to target in your protein of interest? For example `1,2-10` or chain specific `A1-10,B1-20` or entire chains `A`. If left blank, an appropriate site will be selected by the pipeline.
target_hotspot_residues = "A18,A19,A20,A23,A26,A54,A56,A58,A63,A66,A76,A113,A115,A117,A119,A120,A121,A122,A123,A124,A125,A126" # @param {"type":"string","placeholder":""}

# @markdown What is the minimum and maximum size of binders you want to design? Pipeline will randomly sample different sizes between these values.
lengths = "65, 100" # @param {"type":"string","placeholder":"70,150"}

# @markdown How many binder designs passing filters do you require?
number_of_final_designs = 25 # @param {"type":"integer","placeholder":"100"}
# @markdown ---
# @markdown Enter path on your Google drive (/content/drive/MyDrive/BindCraft/[binder_name].json) to previous target settings to continue design campaign. If left empty, it will use the settings above and generate a new settings json in your design output folder.
load_previous_target_settings = "" # @param {"type":"string","placeholder":""}
# @markdown ---

if load_previous_target_settings:
    target_settings_path = load_previous_target_settings
else:
    lengths = [int(x.strip()) for x in lengths.split(',')]

    if len(lengths) != 2:
        raise ValueError("Incorrect specification of binder lengths.")

    settings = {
        "design_path": design_path,
        "binder_name": binder_name,
        "starting_pdb": starting_pdb,
        "chains": chains,
        "target_hotspot_residues": target_hotspot_residues,
        "lengths": lengths,
        "number_of_final_designs": number_of_final_designs
    }

    target_settings_path = os.path.join(design_path, binder_name+".json")
    os.makedirs(design_path, exist_ok=True)

    with open(target_settings_path, 'w') as f:
        json.dump(settings, f, indent=4)

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Binder design settings updated at: {currenttime}")
print(f"New .json file with target settings has been generated in: {target_settings_path}")

Binder design settings updated at: 2026-05-07 04:14:24
New .json file with target settings has been generated in: /content/drive/My Drive/Code/Protein Design/01.5_bindcraft_output/PD1/PD1.json


In [4]:
#@title Advanced settings
# @markdown ---
# @markdown Which binder design protocol to run? Default is recommended. "Beta-sheet" promotes the design of more beta sheeted proteins, but requires more sampling. "Peptide" is optimised for helical peptide binders.
design_protocol = "Default" # @param ["Default","Beta-sheet","Peptide"]
# @markdown What prediction protocol to use?. "Default" performs single sequence prediction of the binder. "HardTarget" uses initial guess to improve complex prediction for difficult targets, but might introduce some bias.
prediction_protocol = "Default" # @param ["Default","HardTarget"]
# @markdown What interface design method to use?. "AlphaFold2" is the default, interface is generated by AlphaFold2. "MPNN" uses soluble MPNN to optimise the interface.
interface_protocol = "AlphaFold2" # @param ["AlphaFold2","MPNN"]
# @markdown What target template protocol to use? "Default" allows for limited amount flexibility. "Masked" allows for greater target flexibility on both sidechain and backbone level.
template_protocol = "Default" # @param ["Default","Masked"]
# @markdown ---

if design_protocol == "Default":
    design_protocol_tag = "default_4stage_multimer"
elif design_protocol == "Beta-sheet":
    design_protocol_tag = "betasheet_4stage_multimer"
elif design_protocol == "Peptide":
    design_protocol_tag = "peptide_3stage_multimer"
else:
    raise ValueError(f"Unsupported design protocol")

if interface_protocol == "AlphaFold2":
    interface_protocol_tag = ""
elif interface_protocol == "MPNN":
    interface_protocol_tag = "_mpnn"
else:
    raise ValueError(f"Unsupported interface protocol")

if template_protocol == "Default":
    template_protocol_tag = ""
elif template_protocol == "Masked":
    template_protocol_tag = "_flexible"
else:
    raise ValueError(f"Unsupported template protocol")

if design_protocol in ["Peptide"]:
    prediction_protocol_tag = ""
else:
    if prediction_protocol == "Default":
        prediction_protocol_tag = ""
    elif prediction_protocol == "HardTarget":
        prediction_protocol_tag = "_hardtarget"
    else:
        raise ValueError(f"Unsupported prediction protocol")

advanced_settings_path = "/content/bindcraft/settings_advanced/" + design_protocol_tag + interface_protocol_tag + template_protocol_tag + prediction_protocol_tag + ".json"

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Advanced design settings updated at: {currenttime}")

Advanced design settings updated at: 2026-05-07 04:14:24


In [5]:
#@title Filters
# @markdown ---
# @markdown Which filters for designs to use? "Default" are recommended, "Peptide" are for the design of peptide binders, "Relaxed" are more permissive but may result in fewer experimental successes, "Peptide_Relaxed" are more permissive filters for non-helical peptides, "None" is for benchmarking.
filter_option = "Default" # @param ["Default", "Peptide", "Relaxed", "Peptide_Relaxed", "None"]
# @markdown ---

if filter_option == "Default":
    filter_settings_path = "/content/bindcraft/settings_filters/default_filters.json"
elif filter_option == "Peptide":
    filter_settings_path = "/content/bindcraft/settings_filters/peptide_filters.json"
elif filter_option == "Relaxed":
    filter_settings_path = "/content/bindcraft/settings_filters/relaxed_filters.json"
elif filter_option == "Peptide_Relaxed":
    filter_settings_path = "/content/bindcraft/settings_filters/peptide_relaxed_filters.json"
elif filter_option == "None":
    filter_settings_path = "/content/bindcraft/settings_filters/no_filters.json"
else:
    raise ValueError(f"Unsupported filter type")

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Filter settings updated at: {currenttime}")

Filter settings updated at: 2026-05-07 04:14:24


# Everything is set, BindCraft is ready to run!

In [6]:
# @title Import functions and settings
from bindcraft.functions import *

args = {"settings":target_settings_path,
        "filters":filter_settings_path,
        "advanced":advanced_settings_path}

# Check if JAX-capable GPU is available, otherwise exit
check_jax_gpu()

# perform checks of input setting files
settings_path, filters_path, advanced_path = (args["settings"], args["filters"], args["advanced"])

### load settings from JSON
target_settings, advanced_settings, filters = load_json_settings(settings_path, filters_path, advanced_path)

settings_file = os.path.basename(settings_path).split('.')[0]
filters_file = os.path.basename(filters_path).split('.')[0]
advanced_file = os.path.basename(advanced_path).split('.')[0]

### load AF2 model settings
design_models, prediction_models, multimer_validation = load_af2_models(advanced_settings["use_multimer_design"])

### perform checks on advanced_settings
bindcraft_folder = "colab"
advanced_settings = perform_advanced_settings_check(advanced_settings, bindcraft_folder)

### generate directories, design path names can be found within the function
design_paths = generate_directories(target_settings["design_path"])

### generate dataframes
trajectory_labels, design_labels, final_labels = generate_dataframe_labels()

trajectory_csv = os.path.join(target_settings["design_path"], 'trajectory_stats.csv')
mpnn_csv = os.path.join(target_settings["design_path"], 'mpnn_design_stats.csv')
final_csv = os.path.join(target_settings["design_path"], 'final_design_stats.csv')
failure_csv = os.path.join(target_settings["design_path"], 'failure_csv.csv')

create_dataframe(trajectory_csv, trajectory_labels)
create_dataframe(mpnn_csv, design_labels)
create_dataframe(final_csv, final_labels)
generate_filter_pass_csv(failure_csv, args["filters"])

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Loaded design functions and settings at: {currenttime}")

Available GPUs:
Tesla T41: gpu
Loaded design functions and settings at: 2026-05-07 04:14:33


In [7]:
for key, value in design_paths.items():
    if not value.startswith('/content/drive/'):
        design_paths[key] = os.path.join('/content/drive', value)
print("Corrected design_paths:")
for key, value in design_paths.items():
    print(f"{key}: {value}")

Corrected design_paths:
Accepted: /content/drive/My Drive/Code/Protein Design/01.5_bindcraft_output/PD1/Accepted
Accepted/Ranked: /content/drive/My Drive/Code/Protein Design/01.5_bindcraft_output/PD1/Accepted/Ranked
Accepted/Animation: /content/drive/My Drive/Code/Protein Design/01.5_bindcraft_output/PD1/Accepted/Animation
Accepted/Plots: /content/drive/My Drive/Code/Protein Design/01.5_bindcraft_output/PD1/Accepted/Plots
Accepted/Pickle: /content/drive/My Drive/Code/Protein Design/01.5_bindcraft_output/PD1/Accepted/Pickle
Trajectory: /content/drive/My Drive/Code/Protein Design/01.5_bindcraft_output/PD1/Trajectory
Trajectory/Relaxed: /content/drive/My Drive/Code/Protein Design/01.5_bindcraft_output/PD1/Trajectory/Relaxed
Trajectory/Plots: /content/drive/My Drive/Code/Protein Design/01.5_bindcraft_output/PD1/Trajectory/Plots
Trajectory/Clashing: /content/drive/My Drive/Code/Protein Design/01.5_bindcraft_output/PD1/Trajectory/Clashing
Trajectory/LowConfidence: /content/drive/My Drive/Cod

In [7]:
#@title Initialise PyRosetta

####################################
####################################
####################################
### initialise PyRosetta
pr.init(f'-ignore_unrecognized_res -ignore_zero_occupancy -mute all -holes:dalphaball {advanced_settings["dalphaball_path"]} -corrections::beta_nov16 true -relax:default_repeats 1')

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python312.ubuntu.cxx11thread.serialization 2026.18+release.5c9f6db69d1a20db6353738e3724092516415bd3 2026-04-30T11:16:34] retrieved from: http://www.pyrosetta.org


In [ ]:
#@title Run BindCraft! - Process Existing Designs
####################################
###################### BindCraft Run - Pre-processing
####################################
import os, time, pandas as pd, numpy as np
from IPython.display import display
from ipywidgets import HTML, VBox

# Helper function to handle Google Drive sync delay
def wait_for_file(filepath, timeout=15):
    start_time = time.time()
    while not os.path.exists(filepath):
        time.sleep(1)
        if time.time() - start_time > timeout:
            return False
    return True

# Colab-specific: live displays
trajectory_csv = os.path.join(target_settings['design_path'], 'trajectory_stats.csv')
final_csv = os.path.join(target_settings['design_path'], 'final_design_stats.csv')

# Ensure CSVs exist or are created for initial read
create_dataframe(trajectory_csv, trajectory_labels)
create_dataframe(final_csv, final_labels)

num_sampled_trajectories = len(pd.read_csv(trajectory_csv)) if os.path.exists(trajectory_csv) and not pd.read_csv(trajectory_csv).empty else 0
num_accepted_designs = len(pd.read_csv(final_csv)) if os.path.exists(final_csv) and not pd.read_csv(final_csv).empty else 0

sampled_trajectories_label = HTML(value=f"<h3 style='color: #1f77b4;'>Sampled Trajectories: {num_sampled_trajectories}</h3>")
accepted_designs_label = HTML(value=f"<h3 style='color: #2ca02c;'>Accepted Designs: {num_accepted_designs}</h3>")
display(VBox([sampled_trajectories_label, accepted_designs_label]))

# Ensure all directories are created
for path in design_paths.values():
    os.makedirs(path, exist_ok=True)

script_start_time = time.time()

# --- NEW PRE-PROCESSING STEP: Process existing unlogged PDBs ---
print("Checking for unlogged PDBs in Trajectory folder and re-processing them...")
existing_trajectory_pdb_files = [f for f in os.listdir(design_paths['Trajectory']) if f.endswith('.pdb')]
logged_design_names = set(pd.read_csv(trajectory_csv)['Design']) if os.path.exists(trajectory_csv) and not pd.read_csv(trajectory_csv).empty else set()

for existing_pdb_filename in existing_trajectory_pdb_files:
    design_name = existing_pdb_filename.split('.pdb')[0]
    if design_name not in logged_design_names:
        print(f"Found unlogged PDB: {design_name}. Re-running hallucination to retrieve metrics...")

        # Extract length and seed from filename (e.g., PD1_l98_s267999)
        parts = design_name.split('_')
        length_str = next((p for p in parts if p.startswith('l')), None)
        seed_str = next((p for p in parts if p.startswith('s')), None)

        try:
            if length_str and seed_str:
                length = int(length_str[1:]) # remove 'l' prefix
                seed = int(seed_str[1:])   # remove 's' prefix
            else:
                raise ValueError("Length or seed prefix not found in filename")
        except (ValueError, IndexError) as e:
            print(f"Could not parse length or seed from filename {design_name}: {e}. Skipping.")
            continue

        helicity_value = load_helicity(advanced_settings) # Re-load helicity
        trajectory_start_time = time.time() # Reset start time for this re-processing

        try:
            # Re-run binder_hallucination to get the 'trajectory' object with 'aux' data
            # This will overwrite the existing PDB but is necessary to get full metrics.
            trajectory = binder_hallucination(design_name, target_settings['starting_pdb'], target_settings['chains'],
                                            target_settings['target_hotspot_residues'], length, seed, helicity_value,
                                            design_models, advanced_settings, design_paths, failure_csv)

            if trajectory.aux['log']['terminate'] == "":
                # Re-define paths as binder_hallucination might re-create them
                trajectory_pdb = os.path.join(design_paths['Trajectory'], f"{design_name}.pdb")
                trajectory_relaxed = os.path.join(design_paths['Trajectory/Relaxed'], f"{design_name}.pdb")

                if not wait_for_file(trajectory_pdb):
                    print(f"Skipping {design_name}: PDB not found after re-hallucination. Potential issue.")
                    continue

                # PyRosetta Relax
                pr_relax(trajectory_pdb, trajectory_relaxed)

                if not wait_for_file(trajectory_relaxed):
                    print(f"Warning: {trajectory_relaxed} sync delay after re-relaxation.")

                binder_chain = 'B'
                num_clashes_relaxed = calculate_clash_score(trajectory_relaxed)
                trajectory_alpha, trajectory_beta, trajectory_loops, trajectory_alpha_interface, trajectory_beta_interface, trajectory_loops_interface, trajectory_i_plddt, trajectory_ss_plddt = calc_ss_percentage(trajectory_pdb, advanced_settings, binder_chain)
                trajectory_interface_scores, trajectory_interface_AA, trajectory_interface_residues = score_interface(trajectory_relaxed, binder_chain)
                trajectory_sequence = trajectory.get_seq(get_best=True)[0]
                traj_seq_notes = validate_design_sequence(trajectory_sequence, num_clashes_relaxed, advanced_settings)
                trajectory_target_rmsd = unaligned_rmsd(target_settings['starting_pdb'], trajectory_pdb, target_settings['chains'], 'A')

                trajectory_metrics = {k: round(v, 2) if isinstance(v, float) else v for k, v in trajectory.aux['log'].items()}
                trajectory_time_text = f"{(time.time() - trajectory_start_time):.0f}s (re-processed)"

                trajectory_data = [design_name, advanced_settings['design_algorithm'], length, seed, helicity_value, target_settings['target_hotspot_residues'], trajectory_sequence, trajectory_interface_residues,
                                    trajectory_metrics.get('plddt'), trajectory_metrics.get('ptm'), trajectory_metrics.get('i_ptm'), trajectory_metrics.get('pae'), trajectory_metrics.get('i_pae'),
                                    trajectory_i_plddt, trajectory_ss_plddt, calculate_clash_score(trajectory_pdb), num_clashes_relaxed, trajectory_interface_scores['binder_score'],
                                    trajectory_interface_scores['surface_hydrophobicity'], trajectory_interface_scores['interface_sc'], trajectory_interface_scores['interface_packstat'],
                                    trajectory_interface_scores['interface_dG'], trajectory_interface_scores['dSASA'], trajectory_interface_scores['interface_dG_SASA_ratio'],
                                    trajectory_interface_scores['interface_fraction'], trajectory_interface_scores['interface_hydrophobicity'], trajectory_interface_scores['interface_nres'], trajectory_interface_scores['interface_interface_hbonds'],
                                    trajectory_interface_scores['interface_hbond_percentage'], trajectory_interface_scores['interface_delta_unsat_hbonds'], trajectory_interface_scores['interface_delta_unsat_hbonds_percentage'],
                                    trajectory_alpha_interface, trajectory_beta_interface, trajectory_loops_interface, trajectory_alpha, trajectory_beta, trajectory_loops, trajectory_interface_AA, trajectory_target_rmsd,
                                    trajectory_time_text, traj_seq_notes, settings_file, filters_file, advanced_file]
                insert_data(trajectory_csv, trajectory_data)
                logged_design_names.add(design_name) # Add to logged set to avoid re-processing if loop runs again

            else:
                print(f"Re-processing {design_name} terminated early: {trajectory.aux['log']['terminate']}")

        except Exception as e:
            print(f"Error re-processing {design_name}: {e}")
            # Optionally log to failure_csv if appropriate

print("Finished checking and re-processing unlogged PDBs.")
# --- END NEW PRE-PROCESSING STEP ---

# Re-read the CSVs to update displayed counts after pre-processing
num_sampled_trajectories = len(pd.read_csv(trajectory_csv)) if os.path.exists(trajectory_csv) and not pd.read_csv(trajectory_csv).empty else 0
num_accepted_designs = len(pd.read_csv(final_csv)) if os.path.exists(final_csv) and not pd.read_csv(final_csv).empty else 0

sampled_trajectories_label.value = f"<h3 style='color: #1f77b4;'>Sampled Trajectories: {num_sampled_trajectories}</h3>"
accepted_designs_label.value = f"<h3 style='color: #2ca02c;'>Accepted Designs: {num_accepted_designs}</h3>"


Checking for unlogged PDBs in Trajectory folder and re-processing them...
Found unlogged PDB: PD1_l72_s822269. Re-running hallucination to retrieve metrics...
Stage 1: Test Logits
1 models [1] recycles 1 hard 0 soft 0.02 temp 1 loss 12.05 helix 1.84 pae 0.78 i_pae 0.77 con 4.68 i_con 3.98 plddt 0.32 ptm 0.61 i_ptm 0.12 rg 11.48
2 models [3] recycles 1 hard 0 soft 0.04 temp 1 loss 9.08 helix 0.94 pae 0.69 i_pae 0.73 con 4.27 i_con 4.14 plddt 0.40 ptm 0.61 i_ptm 0.14 rg 1.66
3 models [4] recycles 1 hard 0 soft 0.05 temp 1 loss 8.91 helix 0.54 pae 0.60 i_pae 0.69 con 3.68 i_con 4.01 plddt 0.52 ptm 0.61 i_ptm 0.14 rg 3.29
4 models [2] recycles 1 hard 0 soft 0.07 temp 1 loss 8.63 helix 0.58 pae 0.59 i_pae 0.66 con 3.56 i_con 4.02 plddt 0.50 ptm 0.61 i_ptm 0.16 rg 2.81
5 models [2] recycles 1 hard 0 soft 0.09 temp 1 loss 8.50 helix 0.74 pae 0.60 i_pae 0.67 con 3.48 i_con 4.09 plddt 0.49 ptm 0.61 i_ptm 0.15 rg 2.49
6 models [4] recycles 1 hard 0 soft 0.11 temp 1 loss 8.29 helix 0.81 pae 0.54 

In [ ]:
#@title Run BindCraft! - Generate New Designs
####################################
###################### BindCraft Run - Main Design Loop
####################################

# Re-read the CSVs to get the latest counts after pre-processing
num_sampled_trajectories = len(pd.read_csv(trajectory_csv)) if os.path.exists(trajectory_csv) and not pd.read_csv(trajectory_csv).empty else 0
num_accepted_designs = len(pd.read_csv(final_csv)) if os.path.exists(final_csv) and not pd.read_csv(final_csv).empty else 0

sampled_trajectories_label.value = f"<h3 style='color: #1f77b4;'>Sampled Trajectories: {num_sampled_trajectories}</h3>"
accepted_designs_label.value = f"<h3 style='color: #2ca02c;'>Accepted Designs: {num_accepted_designs}</h3>"

trajectory_n = num_sampled_trajectories + 1 # Initialize trajectory_n with current count for new designs

### start design loop (now generates NEW designs if needed)
while True:
    if check_accepted_designs(design_paths, mpnn_csv, final_labels, final_csv, advanced_settings, target_settings, design_labels):
        print(f"Target number of accepted designs ({target_settings['number_of_final_designs']}) reached. Stopping design generation.")
        break
    if check_n_trajectories(design_paths, advanced_settings):
        print(f"Max number of trajectories ({advanced_settings['max_trajectories']}) reached. Stopping design generation.")
        break

    trajectory_start_time = time.time()
    seed = int(np.random.randint(0, 999999))
    length = np.random.choice(np.arange(min(target_settings['lengths']), max(target_settings['lengths']) + 1))
    helicity_value = load_helicity(advanced_settings)
    design_name = f"{target_settings['binder_name']}_l{length}_s{seed}"

    # Check if this NEW design_name already exists in the logged set to avoid collision
    if design_name in logged_design_names:
        print(f"Randomly generated design_name {design_name} already exists in logs. Re-generating new random parameters.")
        continue # Skip this iteration and try new random parameters

    print(f"Starting NEW trajectory: {design_name}")
    trajectory = binder_hallucination(design_name, target_settings['starting_pdb'], target_settings['chains'],
                                        target_settings['target_hotspot_residues'], length, seed, helicity_value,
                                        design_models, advanced_settings, design_paths, failure_csv)

    trajectory_pdb = os.path.join(design_paths['Trajectory'], f"{design_name}.pdb")

    # Wait for Hallucination PDB to sync
    if not wait_for_file(trajectory_pdb):
        print(f"Skip: {design_name} PDB sync timeout.")
        continue

    if trajectory.aux['log']['terminate'] == "":
        trajectory_relaxed = os.path.join(design_paths['Trajectory/Relaxed'], f"{design_name}.pdb")

        # PyRosetta Relax
        pr_relax(trajectory_pdb, trajectory_relaxed)

        # Wait for Relaxed PDB to sync
        if not wait_for_file(trajectory_relaxed):
            print(f"Warning: {trajectory_relaxed} sync delay.")

        binder_chain = 'B'
        num_clashes_relaxed = calculate_clash_score(trajectory_relaxed)
        trajectory_alpha, trajectory_beta, trajectory_loops, trajectory_alpha_interface, trajectory_beta_interface, trajectory_loops_interface, trajectory_i_plddt, trajectory_ss_plddt = calc_ss_percentage(trajectory_pdb, advanced_settings, binder_chain)
        trajectory_interface_scores, trajectory_interface_AA, trajectory_interface_residues = score_interface(trajectory_relaxed, binder_chain)
        trajectory_sequence = trajectory.get_seq(get_best=True)[0]
        traj_seq_notes = validate_design_sequence(trajectory_sequence, num_clashes_relaxed, advanced_settings)
        trajectory_target_rmsd = unaligned_rmsd(target_settings['starting_pdb'], trajectory_pdb, target_settings['chains'], 'A')

        trajectory_metrics = {k: round(v, 2) if isinstance(v, float) else v for k, v in trajectory.aux['log'].items()}
        trajectory_time_text = f"{(time.time() - trajectory_start_time):.0f}s"

        trajectory_data = [design_name, advanced_settings['design_algorithm'], length, seed, helicity_value, target_settings['target_hotspot_residues'], trajectory_sequence, trajectory_interface_residues,
                            trajectory_metrics.get('plddt'), trajectory_metrics.get('ptm'), trajectory_metrics.get('i_ptm'), trajectory_metrics.get('pae'), trajectory_metrics.get('i_pae'),
                            trajectory_i_plddt, trajectory_ss_plddt, calculate_clash_score(trajectory_pdb), num_clashes_relaxed, trajectory_interface_scores['binder_score'],
                            trajectory_interface_scores['surface_hydrophobicity'], trajectory_interface_scores['interface_sc'], trajectory_interface_scores['interface_packstat'],
                            trajectory_interface_scores['interface_dG'], trajectory_interface_scores['dSASA'], trajectory_interface_scores['interface_dG_SASA_ratio'],
                            trajectory_interface_scores['interface_fraction'], trajectory_interface_scores['interface_hydrophobicity'], trajectory_interface_scores['interface_nres'], trajectory_interface_scores['interface_interface_hbonds'],
                            trajectory_interface_scores['interface_hbond_percentage'], trajectory_interface_scores['interface_delta_unsat_hbonds'], trajectory_interface_scores['interface_delta_unsat_hbonds_percentage'],
                            trajectory_alpha_interface, trajectory_beta_interface, trajectory_loops_interface, trajectory_alpha, trajectory_beta, trajectory_loops, trajectory_interface_AA, trajectory_target_rmsd,
                            trajectory_time_text, traj_seq_notes, settings_file, filters_file, advanced_file]
        insert_data(trajectory_csv, trajectory_data)
        logged_design_names.add(design_name) # Add new design name to the logged set

    trajectory_n += 1
    sampled_trajectories_label.value = f"<h3 style='color: #1f77b4;'>Sampled Trajectories: {len(pd.read_csv(trajectory_csv))}</h3>"
    accepted_designs_label.value = f"<h3 style='color: #2ca02c;'>Accepted Designs: {len(pd.read_csv(final_csv))}</h3>"

### Verifying Design Progress Files

In [10]:
import os
import pandas as pd

# Define the paths to the CSV files based on your current settings
trajectory_csv = os.path.join(target_settings['design_path'], 'trajectory_stats.csv')
final_csv = os.path.join(target_settings['design_path'], 'final_design_stats.csv')

print(f"Checking CSV files in: {target_settings['design_path']}\n")

# Check trajectory_stats.csv
if os.path.exists(trajectory_csv):
    print(f"Found trajectory_stats.csv. Size: {os.path.getsize(trajectory_csv)} bytes")
    try:
        df_traj = pd.read_csv(trajectory_csv)
        print(f"  - Number of entries in trajectory_stats.csv: {len(df_traj)}")
        if not df_traj.empty:
            display(df_traj.head())
        else:
            print("  - trajectory_stats.csv is empty.")
    except pd.errors.EmptyDataError:
        print("  - trajectory_stats.csv exists but is empty or malformed.")
else:
    print(f"trajectory_stats.csv NOT FOUND at {trajectory_csv}.")

print("\n")

# Check final_design_stats.csv
if os.path.exists(final_csv):
    print(f"Found final_design_stats.csv. Size: {os.path.getsize(final_csv)} bytes")
    try:
        df_final = pd.read_csv(final_csv)
        print(f"  - Number of entries in final_design_stats.csv: {len(df_final)}")
        if not df_final.empty:
            display(df_final.head())
        else:
            print("  - final_design_stats.csv is empty.")
    except pd.errors.EmptyDataError:
        print("  - final_design_stats.csv exists but is empty or malformed.")
else:
    print(f"final_design_stats.csv NOT FOUND at {final_csv}.")

print("\n")

# List contents of key design folders
print("Listing contents of design folders:\n")
for folder_key in ['Trajectory', 'Accepted']:
    folder_path = design_paths.get(folder_key)
    if folder_path and os.path.exists(folder_path):
        print(f"Contents of {folder_path}:")
        files_in_folder = [f for f in os.listdir(folder_path) if f.endswith('.pdb')]
        if files_in_folder:
            for f in files_in_folder[:5]: # Display up to 5 files to avoid overwhelming output
                print(f"  - {f}")
            if len(files_in_folder) > 5:
                print(f"  - ...and {len(files_in_folder) - 5} more PDB files.")
        else:
            print("  - No PDB files found.")
    elif folder_path:
        print(f"Folder NOT FOUND: {folder_path}")
    else:
        print(f"Path for '{folder_key}' not defined in design_paths.")

Checking CSV files in: /content/drive/My Drive/Code/Protein Design/01.5_bindcraft_output/PD1

Found trajectory_stats.csv. Size: 592 bytes
  - Number of entries in trajectory_stats.csv: 0
  - trajectory_stats.csv is empty.


Found final_design_stats.csv. Size: 3890 bytes
  - Number of entries in final_design_stats.csv: 0
  - final_design_stats.csv is empty.


Listing contents of design folders:

Contents of /content/drive/My Drive/Code/Protein Design/01.5_bindcraft_output/PD1/Trajectory:
  - PD1_l78_s355354.pdb
  - PD1_l75_s842094.pdb
  - PD1_l73_s656122.pdb
  - PD1_l99_s774689.pdb
  - PD1_l77_s718512.pdb
  - ...and 149 more PDB files.
Contents of /content/drive/My Drive/Code/Protein Design/01.5_bindcraft_output/PD1/Accepted:
  - No PDB files found.


In [ ]:
import os
import pandas as pd

# Check if GPU is still accessible
print("--- GPU Status ---")
!nvidia-smi

# Check trajectory progress
traj_csv = os.path.join(target_settings['design_path'], 'trajectory_stats.csv')
final_csv = os.path.join(target_settings['design_path'], 'final_design_stats.csv')

if os.path.exists(traj_csv):
    df_traj = pd.read_csv(traj_csv)
    print(f"\nTotal Trajectories sampled: {len(df_traj)}")
else:
    print("\nNo trajectory stats found yet.")

if os.path.exists(final_csv):
    df_final = pd.read_csv(final_csv)
    print(f"Total Accepted Designs: {len(df_final)}")
else:
    print("No accepted designs found yet.")

# List files in the trajectory folder to see physical progress
traj_path = design_paths.get('Trajectory', '')
if os.path.exists(traj_path):
    files = os.listdir(traj_path)
    pdb_count = len([f for f in files if f.endswith('.pdb')])
    print(f"PDB files in Trajectory folder: {pdb_count}")

--- GPU Status ---
/bin/bash: line 1: nvidia-smi: command not found


NameError: name 'target_settings' is not defined

In [ ]:
#@title Consolidate & Rank Designs
#@markdown ---
accepted_binders = [f for f in os.listdir(design_paths["Accepted"]) if f.endswith('.pdb')]

for f in os.listdir(design_paths["Accepted/Ranked"]):
    os.remove(os.path.join(design_paths["Accepted/Ranked"], f))

# load dataframe of designed binders
design_df = pd.read_csv(mpnn_csv)
design_df = design_df.sort_values('Average_i_pTM', ascending=False)

# create final csv dataframe to copy matched rows, initialize with the column labels
final_df = pd.DataFrame(columns=final_labels)

# check the ranking of the designs and copy them with new ranked IDs to the folder
rank = 1
for _, row in design_df.iterrows():
    for binder in accepted_binders:
        target_settings["binder_name"], model = binder.rsplit('_model', 1)
        if target_settings["binder_name"] == row['Design']:
            # rank and copy into ranked folder
            row_data = {'Rank': rank, **{label: row[label] for label in design_labels}}
            final_df = pd.concat([final_df, pd.DataFrame([row_data])], ignore_index=True)
            old_path = os.path.join(design_paths["Accepted"], binder)
            new_path = os.path.join(design_paths["Accepted/Ranked"], f"{rank}_{target_settings['binder_name']}_model{model.rsplit('.', 1)[0]}.pdb")
            shutil.copyfile(old_path, new_path)

            rank += 1
            break

# save the final_df to final_csv
final_df.to_csv(final_csv, index=False)

print("Designs ranked and final_designs_stats.csv generated")

Designs ranked and final_designs_stats.csv generated


In [ ]:
#@title Top 20 Designs
df = pd.read_csv(os.path.join(design_path, 'final_design_stats.csv'))
df.head(20)

In [ ]:
#@title Top Design Display
import py3Dmol
import glob
from IPython.display import HTML

#### pymol top design
top_design_dir = os.path.join(design_path, 'Accepted', 'Ranked')
top_design_pdb = glob.glob(os.path.join(top_design_dir, '1_*.pdb'))[0]

# Visualise in PyMOL
view = py3Dmol.view()
view.addModel(open(top_design_pdb, 'r').read(),'pdb')
view.setBackgroundColor('white')
view.setStyle({'chain':'A'}, {'cartoon': {'color':'#3c5b6f'}})
view.setStyle({'chain':'B'}, {'cartoon': {'color':'#B76E79'}})
view.zoomTo()
view.show()

In [ ]:
#@title Display animation
import glob
from IPython.display import HTML

#### pymol top design
top_design_dir = os.path.join(design_path, 'Accepted', 'Ranked')
top_design_pdb = glob.glob(os.path.join(top_design_dir, '1_*.pdb'))[0]

top_design_name = os.path.basename(top_design_pdb).split('1_', 1)[1].split('_mpnn')[0]
top_design_animation = os.path.join(design_path, 'Accepted', 'Animation', f"{top_design_name}.html")

# Show animation
HTML(top_design_animation)